In [1]:
import pandas as pd
import json
import ollama

print("Biblioteker lastet inn!")


Biblioteker lastet inn!


In [6]:
# De 5 kategoriene vi skal sortere diagnosene i
KATEGORIER = """
1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over
"""

# Last inn rensede data
with open("../data/diagnoses_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Lastet {len(df)} diagnoser")
print(f"\nKategorier:\n{KATEGORIER}")

Lastet 5 diagnoser

Kategorier:

1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over



In [7]:
def klassifiser_diagnose(diagnose_tekst):
    prompt = f"""Du er en medisinsk historiker som kategoriserer historiske diagnosestrenger.

Kategorier:
{KATEGORIER}

Diagnose: "{diagnose_tekst}"

Svar kun med nummeret (1-5) og kategorinavnet. Eksempel: "3. Ex constitutione"
Ikke skriv noe annet."""

    respons = ollama.chat(
        model="gemma4:12b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return respons["message"]["content"].strip()

# Test med én diagnose
test = klassifiser_diagnose("Hysteria med kramper")
print(f"Testresultat: {test}")

Testresultat: 4. Epilepsi-relatert


In [9]:
print("Kategoriserer alle diagnoser...")

resultater = []
for i, (_, rad) in enumerate(df.iterrows()):
    kategori = klassifiser_diagnose(rad["diagnosis_clean"])
    resultater.append(kategori)
    print(f"  [{i+1}/{len(df)}] {rad['id']} → {kategori}")
    
    # Lagre for hver 50. post
    if (i + 1) % 50 == 0:
        df_temp = df.iloc[:i+1].copy()
        df_temp["kategori"] = resultater
        df_temp.to_json("../data/klassifisert_temp.json", 
                        orient="records", force_ascii=False, indent=2)
        print(f"  💾 Lagret fremgang ({i+1} poster)")

df["kategori"] = resultater
print(f"\nFerdig!")

Kategoriserer alle diagnoser...
  [1/5] J0001 → 4. Epilepsi-relatert
  [2/5] J0002 → 3. Ex constitutione
  [3/5] J0003 → 2. Hysterisk personlighet
  [4/5] J0005 → 4. Epilepsi-relatert
  [5/5] J0007 → 5. Annet

Ferdig!


In [10]:
OUTPUT_FILE = "../data/klassifisert.json"
df.to_json(OUTPUT_FILE, orient="records", force_ascii=False, indent=2)
print(f"Lagret {len(df)} klassifiserte diagnoser til {OUTPUT_FILE}")
df

Lagret 5 klassifiserte diagnoser til ../data/klassifisert.json


,id,diagnosis,diagnosis_clean,kategori
0,J0001,Hysteria med kramper,hysteria med kramper,4. Epilepsi-relatert
1,J0002,"ex constitutione, svak konstitusjon","ex constitutione, svak konstitusjon",3. Ex constitutione
2,J0003,Hysterisk personlighet,hysterisk personlighet,2. Hysterisk personlighet
3,J0005,Epilepsi med hysteriske trekk,epilepsi med hysteriske trekk,4. Epilepsi-relatert
4,J0007,ukjent,ukjent,5. Annet
